In [ ]:
# COFEU SFT — Türkçe Instruction Tuning (Kaggle)
# T4 x2 | ~25-30 dk
#
# Bu notebook repo'yu GitHub'dan klonlar (herkese açık) ama
# sft.py'yi Kaggle Input'tan GELEN düzeltilmiş sürümle ezer.
# Böylece çalışma GitHub push durumundan bağımsız olur.

In [ ]:
!git clone --depth 1 https://github.com/cofeu/cofeuai.git /content/cofeuai 2>&1 | tail -2 || true
!pip install -q pyarrow
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os, json, shutil, glob

INPUT_DIRS = [d for d in glob.glob('/kaggle/input/*/*') if os.path.isdir(d)]
if not INPUT_DIRS:
    raise RuntimeError(
        "Kaggle Input bulunamadi. Sag ustte 'Add Input' > 'Upload' ile\n"
        "kaggle_upload/ klasorundeki 3 dosyayi secin:\n"
        "  cofeu.pt, vocab.json, sft.py")

def find_input(name):
    for d in INPUT_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise RuntimeError(
        f"'{name}' Kaggle Input'ta yok. Yuklenecek dosyalar: " +
        ", ".join(os.path.basename(p) for p in INPUT_DIRS))

REPO = '/content/cofeuai'
os.makedirs(f'{REPO}/checkpoints', exist_ok=True)

for name, dest in [('cofeu.pt',  f'{REPO}/checkpoints/cofeu.pt'),
                   ('vocab.json', f'{REPO}/checkpoints/vocab.json'),
                   ('sft.py',     f'{REPO}/python/sft.py')]:
    src = find_input(name)
    shutil.copy(src, dest)
    print(f"{name:12s} <- {src}  ({os.path.getsize(dest)/1e6:.1f} MB)")

print("\nInput klasorleri:", INPUT_DIRS)

In [ ]:
import os, json, glob, shutil

DATA_URL = "https://huggingface.co/datasets/turkish-nlp-suite/InstrucTurca/resolve/main/data/train.jsonl"
DATA_PATH = "/content/turkce_sft.jsonl"

if os.path.exists(DATA_PATH):
    print("veri zaten var, atlaniyor")
else:
    # Kaggle Input'ta donusturulmus veri varsa onu kullan (internet gerekmez)
    local = None
    for p in glob.glob('/kaggle/input/*/*.jsonl'):
        if 'turkce' in os.path.basename(p).lower():
            local = p
            break

    if local:
        print("Kaggle Input'tan aliniyor:", local)
        shutil.copy(local, DATA_PATH)
    else:
        print("veri indiriliyor (390 MB, 2-3 dk)...")
        rc = os.system(f'curl -sL --max-time 900 -o /content/instruk.jsonl "{DATA_URL}"')
        if rc != 0 or not os.path.exists('/content/instruk.jsonl'):
            raise RuntimeError(
                "veri indirilemedi. Kaggle Input'a turkce_sft.jsonl yukleyin "
                "veya notebook'u internet acikken tekrar calistirin.")
        print("donusturuluyor...")
        out, skip = [], 0
        with open('/content/instruk.jsonl', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    o = json.loads(line)
                except Exception:
                    skip += 1
                    continue
                p = (o.get('Output') or '').strip()
                if not p:
                    skip += 1
                    continue
                out.append({'instruction': (o.get('Input') or '').strip(),
                            'input': '', 'output': p})
        with open(DATA_PATH, 'w', encoding='utf-8') as f:
            for r in out:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')
        print(f"donusturulen: {len(out)} | atlanan: {skip}")

with open(DATA_PATH, encoding='utf-8') as f:
    n = sum(1 for line in f if line.strip())
print("ornek sayisi:", n)

In [ ]:
# lr 3e-5: onceki denemelerde 5e-5 bile ezberlemeye yol acti.
# 1 epoch, patience 3: val artinca otomatik durur.
# subprocess kullaniyoruz: egitim basarisiz olursa burada dursun,
# yoksa sonraki hucre "checkpoint yok" hatasiyla asil nedeni gizler.
import subprocess, sys

cmd = [
    sys.executable, "python/sft.py",
    "--data", "/content/turkce_sft.jsonl",
    "--resume", "/content/cofeuai/checkpoints/cofeu.pt",
    "--out-dir", "/content/cofeuai/checkpoints/sft_tr",
    "--epochs", "1", "--max-samples", "20000",
    "--batch-size", "16", "--lr", "3e-5", "--patience", "3",
    "--eval-interval", "200", "--save-interval", "500",
    "--log-interval", "50", "--device", "cuda",
]
print("calistiriliyor:", " ".join(cmd), "\n")
r = subprocess.run(cmd, cwd="/content/cofeuai")
if r.returncode != 0:
    raise RuntimeError(f"SFT egitimi basarisiz, exit={r.returncode}")
print("\nEGITIM TAMAM")

In [ ]:
import sys, torch
sys.path.insert(0, '/content/cofeuai/python')
from model import CofeuTransformer, ModelConfig
from tokenizer import BPETokenizer
import sft

CKPT = '/content/cofeuai/checkpoints/sft_tr/cofeu_sft.pt'
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
print("iteration:", ck.get('iteration'), "| val:", ck.get('val_loss'))

cfg = ModelConfig(**ck['config'].__dict__)
m = CofeuTransformer(cfg)
m.load_state_dict(ck['model_state'])
m.eval().cuda()
tok = BPETokenizer.load('/content/cofeuai/checkpoints/vocab.json')

for q in ("Merhaba, nasılsın?",
          "Türkiye'nin başkenti neresidir?",
          "Fotosentez nasıl çalışır?",
          "Bana kısa bir motivasyon cümlesi söyle."):
    p, _ = sft.build_text(q, "", "")
    x = torch.tensor([tok.encode(p)], dtype=torch.long).cuda()
    torch.manual_seed(42)
    out = m.generate(x, 120, temperature=0.7, top_k=40, eos_token_id=tok.eos_id)
    print(f"\n>>> {q}\n{tok.decode(out[0, len(x[0]):].tolist())}")

In [ ]:
!cd /content/cofeuai && zip -j /content/cofeu_sft_tr.zip checkpoints/sft_tr/cofeu_sft.pt && ls -la /content/cofeu_sft_tr.zip
# Indirme: Kaggle sag ustte "Save Version" > Output sekmesi > cofeu_sft_tr.zip